# IBKR NQ Globex validation v0.2.0
Uses existing NQ quarterly consolidated Parquets; never downloads IBKR data or modifies source. Start with pilot, then run full nine-contract audit.

In [ ]:
import importlib, subprocess, sys
from pathlib import Path

REPO = Path("/content/IBKR-market-data-cleaning-pipeline")
URL = "https://github.com/hh4832/IBKR-market-data-cleaning-pipeline.git"
if (REPO / ".git").is_dir():
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    if origin.rstrip("/").removesuffix(".git") != URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError("Unexpected Git remote; aborting")
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
        raise RuntimeError("Uncommitted changes in checkout; refusing to overwrite")
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
elif REPO.exists():
    raise RuntimeError("Existing directory is not a Git checkout")
else:
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", str(REPO)+"[colab]"], check=True)
src = str(REPO / "src")
if src in sys.path:
    sys.path.remove(src)
sys.path.insert(0, src)
for name in list(sys.modules):
    if name == "ibkr_qc" or name.startswith("ibkr_qc."):
        del sys.modules[name]
importlib.invalidate_caches()
from ibkr_qc.nq_pipeline import NQRunConfig, run_nq_qc
print("HEAD:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short=12", "HEAD"], text=True).strip())

## Authenticate Drive and reuse the prior Pilot folder as a location reference
The previous run folder is **not** a cache and must never be overwritten. The notebook queries its actual parent folder ID using the Drive API and uploads the next run as a sibling. Existing cache stays in MyDrive. If you move the cache, update CACHE_DIR separately.

In [ ]:
from google.colab import drive
from pathlib import Path
from ibkr_qc.drive_io import authorized_drive

drive.mount("/content/drive")
service = authorized_drive()
SOURCE_FOLDER_ID = "1C0wfYpT8zioRmZxBnqO7d79-zJyD_S5w"

# Existing successful PILOT REPORT (read-only): do not upload into this immutable run.
# The new QA run is created as its sibling, under the verified Drive parent.
PILOT_REPORT_URL = "https://drive.google.com/drive/folders/1yJyohYT6NGTh6QlrtXsu1FeMeAQox3me"
PILOT_REPORT_ID = PILOT_REPORT_URL.split("/folders/", 1)[1].split("?", 1)[0]
pilot_folder = service.files().get(
    fileId=PILOT_REPORT_ID, fields="id,name,mimeType,parents",
    supportsAllDrives=True
).execute()
if pilot_folder.get("mimeType") != "application/vnd.google-apps.folder" or "_NQ_GLOBEX_PILOT" not in pilot_folder.get("name", ""):
    raise RuntimeError("Provided Drive URL is not the known completed NQ Pilot run folder")
parents = pilot_folder.get("parents") or []
if len(parents) != 1:
    raise RuntimeError("Cannot safely determine the parent folder of NQ Pilot")
OUTPUT_PARENT_ID = parents[0]  # NOT PILOT_REPORT_ID; keep each archived run immutable.
parent_info = service.files().get(
    fileId=OUTPUT_PARENT_ID, fields="id,name,mimeType", supportsAllDrives=True
).execute()
if parent_info.get("mimeType") != "application/vnd.google-apps.folder":
    raise RuntimeError("NQ report parent is not a Drive folder")
if OUTPUT_PARENT_ID in (SOURCE_FOLDER_ID, PILOT_REPORT_ID):
    raise RuntimeError("Unsafe output parent; refusing to overwrite existing data")

MAX_CONTRACTS = 1  # PILOT only; change to 0 to validate ALL nine contracts.
TOLERANCE_MINUTES = 2
# Cache stays at its original MyDrive path so the prior download is reused.
# If you later MOVE the cache, change this filesystem path to its NEW MyDrive location.
CACHE_DIR = Path("/content/drive/MyDrive/IBKR_NQ_GLOBEX_QC_CACHE_V020")
OUTPUT_DIR = Path("/content/ibkr_nq_qc_outputs")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Original NQ source:", SOURCE_FOLDER_ID)
print("Previous Pilot (read-only):", PILOT_REPORT_URL)
print("New runs under:", parent_info["name"], OUTPUT_PARENT_ID)
print("Cache path:", CACHE_DIR)
print("Mode:", "FULL" if MAX_CONTRACTS==0 else "PILOT")


## Run QA
CME special holidays/halts and IBKR bar-label convention remain unverified. Per-contract only: no rollover selection or backadjustment. research_ready is always false in this validation phase.

In [ ]:
from ibkr_qc.nq_pipeline import NQRunConfig, run_nq_qc
import json
import pandas as pd

cfg = NQRunConfig(source_folder_id=SOURCE_FOLDER_ID,
    output_parent_id=OUTPUT_PARENT_ID, max_contracts=MAX_CONTRACTS,
    tolerance_minutes=TOLERANCE_MINUTES, upload=True).validate()
run_dir = run_nq_qc(service, cfg, cache_root=CACHE_DIR, output_root=OUTPUT_DIR)
info = json.loads((run_dir/"run_info.json").read_text(encoding="utf-8"))
print({k:info[k] for k in (
    "mode", "status", "total_contract_folders", "successful_contracts",
    "globex_expected_standard_minutes", "possible_missing_standard_minutes",
    "taipei_candidate_anchor_rows", "research_ready",
)})
display(pd.read_csv(run_dir/"daily_globex_coverage.csv").head(10))
display(pd.read_csv(run_dir/"taipei_candidate_anchor_prices.csv").head(10))
print("Local outputs:",run_dir)